# Lab 5 - Bernstein-Vazirani Algorithm

In this practical session, we will implement one of the most famous quantum algorithms: the Bernstein-Vazirani algorithm.

The goal is to discover an unknown binary string hidden inside an oracle.

This algorithm highlights how a quantum computer can extract information more efficiently than a classical approach.

<figure>
    <img src="bernstein_vazirani.jpeg"
    <figcaption><em>Figure 1 - Generic circuit of the Bernstein-Vazirani algorithm.</em></figcaption>
</figure>

## Problem Statement

Assume that a secret binary string

$$s = s_0 s_1 \dots s_{n-1}$$

is hidden in an oracle.

The oracle computes:

$$f(x) = s \cdot x \pmod{2}$$

where "·" denotes the bitwise inner product.

### Example

If:

$s = 1011$

and

$x = 1101$

then:

\begin{align*}
f(x) &= 1 \times 1 + 0 \times 1 + 1 \times 0 + 1 \times 1 \pmod{2} \\
     &= 2 \pmod{2} \\
     &= 0
\end{align*}

The objective is to recover the secret string.

## Discussion

### Question 1

How many oracle evaluations are required by a classical algorithm to recover a secret string of n bits?

### Question 2

How many oracle evaluations do you expect from the quantum algorithm?

<details>
<summary>Show solution</summary>

1. For a secret string of $n$ bits, a classical algorithm requires $n$ oracle evaluations. Each query reveals at most one bit of information.

2. Only 1 oracle evaluation is required. The complete secret string is recovered with a single query.

## Library import

In [ ]:
#TODO: Import necessary modules and get the default QPU

...

## Building the Oracle

The Bernstein-Vazirani oracle implements the function

$f(x) = s \cdot x \pmod{2}$

where:

- $s$ is the hidden binary string,
- $x$ is the input bit string,
- the product is computed modulo 2.

The oracle operates on:

- a register containing the input bits,
- an auxiliary qubit used to store the result of the oracle evaluation.

### Objective

Implement a function that creates the oracle associated with a given secret string.

Your function should:

1. Create the required quantum register.
2. Encode the secret string into the oracle.
3. Return the corresponding quantum circuit.

The returned circuit must be compatible with the remainder of the notebook.

### Example

For:

secret_string = "1011"

the oracle should implement:

$f(x) = x_0 \oplus x_2 \oplus x_3$

### Before Coding

Consider the following questions:

1. How many qubits are required to implement the oracle for an n-bit secret string?
2. Why is an auxiliary qubit needed?
3. Which gate introduced in the previous practical sessions could be used to implement the XOR operation?

<details>
<summary>Show solution</summary>

1. We need n qubits to represent the input register $x$, and 1 auxiliary qubit to store the oracle output.<br>
   Therefore, the oracle requires $n+1$ qubits in total.

2. Quantum oracles are usually implemented as reversible operations.<br>
   Rather than computing:<br>
   $$f(x)$$

   directly, the oracle performs:

   $$|x\rangle |y\rangle \longrightarrow |x\rangle |y \oplus f(x)\rangle$$

   The auxiliary qubit stores the result of the function evaluation.

   Without this additional qubit, the operation would not be reversible and therefore could not be implemented as a valid quantum operation.<br>
   In the Bernstein-Vazirani algorithm, the auxiliary qubit will later be prepared in a special state that allows the function value to be encoded as a quantum phase.

3. The required gate is the CNOT gate.<br>
   Recall that a CNOT performs:
   $$y \rightarrow y \oplus x$$

   where:

   - the control qubit contains $x$,
   - the target qubit contains $y$.

   The target qubit is flipped only when the control qubit is equal to 1.<br>
   Since the Bernstein-Vazirani function is based on sums modulo 2:
   $$f(x)=s\cdot x \pmod 2$$

   and modulo-2 addition is exactly the XOR operation, CNOT gates are the natural building blocks for implementing the oracle.

In [ ]:
def bv_oracle(secret_string):
    """
    Build and return the Bernstein-Vazirani oracle
    associated with the provided secret string.
    """

    # Your code here
    ...

## Test your Oracle

Use the secret string:

1011

and display the resulting circuit.

Can you identify the control qubits used by the oracle?

In [ ]:
#TODO: Define the secret string, create and display the Bernstein-Vazirani oracle

...

NameError: name 'bv_oracle' is not defined

## Preparing the Input State

Before interacting with the oracle, the Bernstein-Vazirani algorithm prepares a very specific quantum state.

The objective is to create:

$$
|+\rangle^{\otimes n} \otimes |-\rangle
$$

where:

$$
|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}
$$

and

$$
|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}
$$

### Questions

Before writing any code:

1. Which qubit should contain the state $|-\rangle$ ?
2. Why do we prepare the qubits in superposition before calling the oracle?

### Objective

Create a circuit preparing the state:

$$
|+\rangle^{\otimes n} \otimes |-\rangle
$$

In [ ]:
n = len(secret_string)

prep_prog = Program()

qubits = prep_prog.qalloc(n + 1)

# TODO:
# Prepare the state |+>^n ⊗ |->
# Generate the circuit and display it

### Question

Why must the auxiliary qubit be initialised in |1⟩ before the Hadamard gate is applied?

What would happen if it remained in |0⟩?

<details>
<summary>Show solution</summary>

1. After the Hadamard gate it becomes $|−\rangle = (|0\rangle − |1\rangle)/\sqrt{2}$ which enables phase kickback.

2. The algorithm no longer encodes the secret string in the phase, so the Bernstein-Vazirani algorithm does not work correctly.


## Revealing the Secret through Interference

After the oracle has been applied, the secret string is encoded in the phase of the quantum state.

To extract this information, we must exploit quantum interference.

The final layer of Hadamard gates transforms these phase differences into measurable amplitudes:

- Constructive interference amplifies the state corresponding to the hidden string.
- Destructive interference cancels the amplitudes of the other states.

Recall that:

$$
H^2 = I
$$

but here the second Hadamard layer does much more than simply "undo" the first one: it converts phase information into measurement outcomes.

### Objective

Apply the transformation required to reveal the hidden secret string.

In [ ]:
recovery_prog = Program()

qubits = recovery_prog.qalloc(n + 1)

# TODO:
# Apply the required gates
# Which qubits should receive a Hadamard gate?

recovery_circuit = recovery_prog.to_circ()

## Assemble the Full Circuit

The complete Bernstein-Vazirani circuit is composed of:

- State preparation
- Oracle
- Final Hadamard layer

Predict the measurement result before executing the circuit.

What binary value do you expect to obtain?

In [ ]:
#TODO: Combine the preparation circuit, oracle, and recovery circuit to form the full Bernstein-Vazirani circuit
# and display it

...

## Execute the Algorithm

Run the circuit and compare the measurement results with the secret string.

Do they match?

In [ ]:
#TODO: Submit the Bernstein-Vazirani circuit to the quantum processor and display the results

...

## Challenge

Modify the secret string and run the algorithm again.

Try:

- 1100
- 1001
- 1111

Can you recover the hidden string with a single execution?

## Key Takeaways

In this practical session you have:

✅ Built a quantum oracle

✅ Used superposition through Hadamard gates

✅ Used CNOT gates to encode information

✅ Recovered a secret string with a single oracle call

The Bernstein-Vazirani algorithm is one of the simplest examples showing how quantum interference can be exploited to extract information efficiently.